# TamerLite Tutorial

**TamerLite** is a heuristic search-based planner integrated with the
[Unified Planning (UP)](https://github.com/aiplan4eu/unified-planning) framework.
It targets **classical**, **numeric**, **temporal**, and **temporal-numeric**
planning problems — durative actions over continuous time with integer / real
fluents, in any combination.

This notebook walks through every configuration option with runnable examples.

## Installation

The recommended setup uses [uv](https://github.com/astral-sh/uv):

```bash
uv sync --all-extras    # creates .venv, builds rustamer, installs deps
```

Or, with pip:

```bash
pip install crates/rustamer   # build the Rust extension (requires Rust toolchain + maturin)
pip install .                 # install tamerlite
```

> **No Rust toolchain?** Set `DISABLE_RUSTAMER=true` before launching Jupyter.
> TamerLite automatically falls back to a pure-Python implementation.

## Setup

Run the cell below to import Unified Planning, register the TamerLite engine,
and suppress boilerplate output.

In [1]:
import time
import warnings

warnings.filterwarnings("ignore")

from unified_planning.engines import PlanGenerationResultStatus
from unified_planning.plans import PlanKind
from unified_planning.shortcuts import *

from tamerlite import HeuristicParams, MultiqueueParams, SearchParams
from tamerlite.engine import TamerLite

# Register TamerLite with Unified Planning
env = get_environment()
env.factory.add_engine("tamerlite", "tamerlite.engine", "TamerLite")
env.credits_stream = None  # suppress credits header

print("TamerLite registered.")

TamerLite registered.


### Supported problem features

`TamerLite.supported_kind()` lists every problem feature the engine handles.
Use `TamerLite.supports(problem.kind)` to gate a specific problem programmatically.

In [2]:
# Inspect the full set of supported problem features
sk = TamerLite.supported_kind()
print(sk)

# To check programmatically whether a specific problem is supported:
#   TamerLite.supports(problem.kind)  →  True / False

PROBLEM_CLASS: ['ACTION_BASED']
PROBLEM_TYPE: ['GENERAL_NUMERIC_PLANNING', 'SIMPLE_NUMERIC_PLANNING']
TIME: ['INTERMEDIATE_CONDITIONS_AND_EFFECTS', 'DURATION_INEQUALITIES', 'CONTINUOUS_TIME']
EXPRESSION_DURATION: ['STATIC_FLUENTS_IN_DURATIONS', 'FLUENTS_IN_DURATIONS', 'INT_TYPE_DURATIONS']
NUMBERS: ['CONTINUOUS_NUMBERS', 'DISCRETE_NUMBERS']
CONDITIONS_KIND: ['UNIVERSAL_CONDITIONS', 'DISJUNCTIVE_CONDITIONS', 'NEGATIVE_CONDITIONS', 'EXISTENTIAL_CONDITIONS', 'EQUALITIES']
EFFECTS_KIND: ['DECREASE_EFFECTS', 'FLUENTS_IN_OBJECT_ASSIGNMENTS', 'FLUENTS_IN_NUMERIC_ASSIGNMENTS', 'FLUENTS_IN_BOOLEAN_ASSIGNMENTS', 'STATIC_FLUENTS_IN_NUMERIC_ASSIGNMENTS', 'STATIC_FLUENTS_IN_OBJECT_ASSIGNMENTS', 'INCREASE_EFFECTS', 'STATIC_FLUENTS_IN_BOOLEAN_ASSIGNMENTS']
TYPING: ['FLAT_TYPING', 'HIERARCHICAL_TYPING']
PARAMETERS: ['BOUNDED_INT_FLUENT_PARAMETERS', 'BOUNDED_INT_ACTION_PARAMETERS', 'BOOL_FLUENT_PARAMETERS', 'BOOL_ACTION_PARAMETERS']
FLUENTS_TYPE: ['REAL_FLUENTS', 'NUMERIC_FLUENTS', 'OBJECT_FLUENTS', 'I

### Helpers

Two helpers used throughout this notebook:

- **`solve_and_summarize(problem, params, ...)`** — run `OneshotPlanner` and
  return a compact dict: `status`, `expanded` states, `depth`, `plan_len`, `time_s`.
- **`show_table(rows)`** — print a list of dicts as a fixed-width table.

In [3]:
# ── Reusable helpers ──────────────────────────────────────────────────────────


def solve_and_summarize(
    problem, params=None, heuristic_fn=None, label=None, timeout=60
):
    """Run OneshotPlanner and return a compact summary dict."""
    kwargs = {"params": {"search": params}} if params is not None else {}
    t0 = time.time()
    with OneshotPlanner(name="tamerlite", **kwargs) as planner:
        res = planner.solve(problem, heuristic=heuristic_fn, timeout=timeout)
    elapsed = time.time() - t0

    if res.plan is None:
        plan_len = None
    elif res.plan.kind == PlanKind.SEQUENTIAL_PLAN:
        plan_len = len(res.plan.actions)
    else:
        plan_len = len(res.plan.timed_actions)

    return {
        "label": label or "default",
        "status": res.status.name,
        "expanded": res.metrics.get("expanded_states", "?") if res.metrics else "?",
        "depth": res.metrics.get("goal_depth", "?") if res.metrics else "?",
        "plan_len": plan_len,
        "time_s": f"{elapsed:.3f}",
    }


def show_table(rows, cols=None):
    """Print a list of dicts as a fixed-width table."""
    if not rows:
        print("(no rows)")
        return
    if cols is None:
        cols = list(rows[0].keys())
    widths = {c: max(len(c), max(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    fmt = " | ".join(f"{{:<{widths[c]}}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(fmt.format(*cols))
    print(sep)
    for r in rows:
        print(fmt.format(*[str(r.get(c, "")) for c in cols]))

## 1 · Example Problems

We build three small problems used throughout the notebook: a numeric **Warehouse**,
a numeric **Counters** problem (fluent-dependent effects), and a temporal
**Flight**. Each is the natural testbed for different TamerLite options in §3.

### 1.1 · Warehouse (numeric)

TamerLite handles classical and numeric planning problems built from
`InstantaneousAction`. We use a small **Warehouse** domain:

| Fluent | Type | Initial | Meaning |
|--------|------|---------|---------|
| `stock` | `IntType()` | 0 | Items in inventory |
| `cash` | `RealType()` | 20 | Available money |

| Action | Precondition | Effects |
|--------|-------------|---------|
| `produce` | `cash ≥ 3` | `stock += 1`, `cash -= 3` |
| `sell` | `stock ≥ 1` | `stock -= 1`, `cash += 5` |

**Goal**: `stock ≥ 6 ∧ cash ≥ 12`

The goal can't be satisfied by `produce` alone (it drains cash) nor `sell` alone (no stock to sell). A valid plan must interleave both — small enough to read end-to-end, large enough to make heuristic and search-algorithm differences visible in §3.

In [4]:
# ── 1.1  Define the Warehouse problem ─────────────────────────────────────────
stock = Fluent("stock", IntType())
cash = Fluent("cash", RealType())

produce = InstantaneousAction("produce")
produce.add_precondition(GE(cash, 3))
produce.add_increase_effect(stock, 1)
produce.add_decrease_effect(cash, 3)

sell = InstantaneousAction("sell")
sell.add_precondition(GE(stock, 1))
sell.add_decrease_effect(stock, 1)
sell.add_increase_effect(cash, 5)

warehouse = Problem("warehouse")
warehouse.add_fluent(stock, default_initial_value=0)
warehouse.add_fluent(cash, default_initial_value=20)
warehouse.add_actions([produce, sell])
warehouse.add_goal(GE(stock, 6))
warehouse.add_goal(GE(cash, 12))

assert TamerLite.supports(warehouse.kind), "Problem kind not supported"
print(warehouse)

problem name = warehouse

fluents = [
  integer stock
  real cash
]

actions = [
  action produce {
    preconditions = [
      (3 <= cash)
    ]
    effects = [
      stock += 1
      cash -= 3
    ]
  }
  action sell {
    preconditions = [
      (1 <= stock)
    ]
    effects = [
      stock -= 1
      cash += 5
    ]
  }
]

initial fluents default = [
  integer stock := 0
  real cash := 20
]

initial values = [
]

goals = [
  (6 <= stock)
  (12 <= cash)
]




### 1.2 · Counters (numeric, fluent-dependent effects)

The **fo-counters** domain (IPC 2023 numeric track) features actions whose
numeric effects depend on other fluents — e.g. `increment(c)` adds `rate_value(c)`
to `value(c)`. We use it in §3.1.1 to show what `inadmissible_numeric_heuristic_variant` actually does.

| Fluent | Type | Initial | Meaning |
|--------|------|---------|---------|
| `value(c)` | `IntType()` | 0 | Counter `c`'s current value |
| `rate_value(c)` | `IntType()` | 0 | How much `increment`/`decrement` change `value(c)` |

| Action | Precondition | Effect |
|--------|--------------|--------|
| `increment(c)` | `value(c) + rate_value(c) ≤ 10` | `value(c) += rate_value(c)` *(complex)* |
| `decrement(c)` | `value(c) − rate_value(c) ≥ 0` | `value(c) -= rate_value(c)` *(complex)* |
| `increase_rate(c)` | `rate_value(c) + 1 ≤ 10` | `rate_value(c) += 1` |
| `decrease_rate(c)` | `rate_value(c) − 1 ≥ 0` | `rate_value(c) -= 1` |

Four counters `c0…c3`. **Goal**: `value(c_i) + 1 ≤ value(c_{i+1})` for each adjacent pair — forcing a strictly increasing sequence of counter values.

In [5]:
# ── 1.2  Define the fo-counters problem ───────────────────────────────────────
counter = UserType("counter")
value = Fluent("value", IntType(), c=counter)
rate_value = Fluent("rate_value", IntType(), c=counter)

increment = InstantaneousAction("increment", c=counter)
increment.add_precondition(LE(Plus(value(increment.c), rate_value(increment.c)), 10))
increment.add_increase_effect(value(increment.c), rate_value(increment.c))

decrement = InstantaneousAction("decrement", c=counter)
decrement.add_precondition(GE(Minus(value(decrement.c), rate_value(decrement.c)), 0))
decrement.add_decrease_effect(value(decrement.c), rate_value(decrement.c))

increase_rate = InstantaneousAction("increase_rate", c=counter)
increase_rate.add_precondition(LE(Plus(rate_value(increase_rate.c), 1), 10))
increase_rate.add_increase_effect(rate_value(increase_rate.c), 1)

decrease_rate = InstantaneousAction("decrease_rate", c=counter)
decrease_rate.add_precondition(GE(Minus(rate_value(decrease_rate.c), 1), 0))
decrease_rate.add_decrease_effect(rate_value(decrease_rate.c), 1)

counter_objects = [Object(f"c{i}", counter) for i in range(4)]
counters_problem = Problem("fo-counters")
counters_problem.add_fluent(value, default_initial_value=0)
counters_problem.add_fluent(rate_value, default_initial_value=0)
counters_problem.add_objects(counter_objects)
counters_problem.add_actions([increment, decrement, increase_rate, decrease_rate])
for i in range(len(counter_objects) - 1):
    counters_problem.add_goal(
        LE(Plus(value(counter_objects[i]), 1), value(counter_objects[i + 1]))
    )

assert TamerLite.supports(counters_problem.kind), "Problem kind not supported"
print(counters_problem)

problem name = fo-counters

types = [counter]

fluents = [
  integer value[c=counter]
  integer rate_value[c=counter]
]

actions = [
  action increment(counter c) {
    preconditions = [
      ((value(c) + rate_value(c)) <= 10)
    ]
    effects = [
      value(c) += rate_value(c)
    ]
  }
  action decrement(counter c) {
    preconditions = [
      (0 <= (value(c) - rate_value(c)))
    ]
    effects = [
      value(c) -= rate_value(c)
    ]
  }
  action increase_rate(counter c) {
    preconditions = [
      ((rate_value(c) + 1) <= 10)
    ]
    effects = [
      rate_value(c) += 1
    ]
  }
  action decrease_rate(counter c) {
    preconditions = [
      (0 <= (rate_value(c) - 1))
    ]
    effects = [
      rate_value(c) -= 1
    ]
  }
]

objects = [
  counter: [c0, c1, c2, c3]
]

initial fluents default = [
  integer value[c=counter] := 0
  integer rate_value[c=counter] := 0
]

initial values = [
]

goals = [
  ((value(c0) + 1) <= value(c1))
  ((value(c1) + 1) <= value(c2))
  ((value

### 1.3 · Temporal Flight (temporal)

Eight cities A–H connected by a sparse graph; some edges have an **express**
variant in addition to the regular slow connection:

```
A ── B           E ── G
│    │           │    │
└─ C ┴─ D ─── F ─┴── H
```

Slow edges (use `fly`, duration 10): A-B, A-C, B-D, C-D, C-E, D-F, E-G, F-H, G-H
Express edges (use `fly_express`, duration 5): A-C, C-D, D-F, F-H

| Action | Duration | Precondition | Effect |
|--------|----------|--------------|--------|
| `fly` | 10 | `at(l_from)` and `connected(l_from, l_to)` | depart at start, arrive at end |
| `fly_express` | 5 | `at(l_from)` and `express_route(l_from, l_to)` | depart at start, arrive at end |

**Initial**: `at(A)`. **Goal**: `at(H)`.

The optimal makespan is 20 (the all-express bottom path A→C→D→F→H, 4 hops × 5);
the slow-only paths take 40 (4 hops × 10). This gap is what §4 Anytime exploits.

In [6]:
# ── 1.3  Define the Temporal Flight problem ───────────────────────────────────
City = UserType("City")

at_city = Fluent("at", BoolType(), city=City)
connected = Fluent("connected", BoolType(), l_from=City, l_to=City)
express_route = Fluent("express_route", BoolType(), l_from=City, l_to=City)

# Slow fly: duration 10, needs a regular connection.
fly = DurativeAction("fly", l_from=City, l_to=City)
fly.set_fixed_duration(10)
fly.add_condition(StartTiming(), at_city(fly.l_from))
fly.add_condition(StartTiming(), connected(fly.l_from, fly.l_to))
fly.add_effect(StartTiming(), at_city(fly.l_from), False)
fly.add_effect(EndTiming(), at_city(fly.l_to), True)

# Express fly: duration 5, only on edges marked express_route.
fly_express = DurativeAction("fly_express", l_from=City, l_to=City)
fly_express.set_fixed_duration(5)
fly_express.add_condition(StartTiming(), at_city(fly_express.l_from))
fly_express.add_condition(
    StartTiming(), express_route(fly_express.l_from, fly_express.l_to)
)
fly_express.add_effect(StartTiming(), at_city(fly_express.l_from), False)
fly_express.add_effect(EndTiming(), at_city(fly_express.l_to), True)

A, B, C, D, E, F, G, H = [
    Object(n, City) for n in ["A", "B", "C", "D", "E", "F", "G", "H"]
]

temporal_flight = Problem("temporal_flight")
temporal_flight.add_fluent(at_city, default_initial_value=False)
temporal_flight.add_fluent(connected, default_initial_value=False)
temporal_flight.add_fluent(express_route, default_initial_value=False)
temporal_flight.add_actions([fly, fly_express])
temporal_flight.add_objects([A, B, C, D, E, F, G, H])

temporal_flight.set_initial_value(at_city(A), True)
slow_edges = [(A, B), (A, C), (B, D), (C, D), (C, E), (D, F), (E, G), (F, H), (G, H)]
express_edges = [(A, C), (C, D), (D, F), (F, H)]
for src, dst in slow_edges:
    temporal_flight.set_initial_value(connected(src, dst), True)
    temporal_flight.set_initial_value(connected(dst, src), True)
for src, dst in express_edges:
    temporal_flight.set_initial_value(express_route(src, dst), True)
    temporal_flight.set_initial_value(express_route(dst, src), True)
temporal_flight.add_goal(at_city(H))

assert TamerLite.supports(temporal_flight.kind), "Problem kind not supported"
print(temporal_flight)

problem name = temporal_flight

types = [City]

fluents = [
  bool at[city=City]
  bool connected[l_from=City, l_to=City]
  bool express_route[l_from=City, l_to=City]
]

actions = [
  durative action fly(City l_from, City l_to) {
    duration = [10, 10]
    conditions = [
      [start]:
        at(l_from)
        connected(l_from, l_to)
    ]
    effects = [
      start:
        at(l_from) := false:
      end:
        at(l_to) := true:
    ]
    simulated effects = [
    ]
  }
  durative action fly_express(City l_from, City l_to) {
    duration = [5, 5]
    conditions = [
      [start]:
        at(l_from)
        express_route(l_from, l_to)
    ]
    effects = [
      start:
        at(l_from) := false:
      end:
        at(l_to) := true:
    ]
    simulated effects = [
    ]
  }
]

objects = [
  City: [A, B, C, D, E, F, G, H]
]

initial fluents default = [
  bool at[city=City] := false
  bool connected[l_from=City, l_to=City] := false
  bool express_route[l_from=City, l_to=City] := f

## 2 · Default / Basic Usage

With no params, TamerLite uses its built-in defaults: `wastar` + `hff` + weight `0.8`
(equivalent to `SearchParams(search="wastar", heuristic="hff", weight=0.8)`).
The resulting plan is validated with `PlanValidator`.

Numeric (Warehouse):

In [7]:
with OneshotPlanner(name="tamerlite") as planner:
    res = planner.solve(warehouse)

print("Status :", res.status.name)
print("Plan   :", res.plan)
print("Metrics:", res.metrics)

# Validate the plan
with PlanValidator(problem_kind=warehouse.kind) as v:
    print("Valid  :", bool(v.validate(warehouse, res.plan)))

Status : SOLVED_SATISFICING
Plan   : SequentialPlan:
    produce
    produce
    sell
    produce
    produce
    sell
    produce
    produce
    sell
    produce
    sell
    produce
    produce
    produce
    produce
    sell
Metrics: {'goal_depth': '16', 'expanded_states': '50'}
Valid  : True


Temporal (Flight) — the plan is a `TimeTriggeredPlan` with
`(start_time, action_instance, duration)` triples:

In [8]:
with OneshotPlanner(name="tamerlite") as planner:
    res = planner.solve(temporal_flight)

print("Status   :", res.status.name)
print("Plan kind:", res.plan.kind)
print()
print("Schedule (sorted by start time):")
for t_start, action, duration in sorted(res.plan.timed_actions, key=lambda x: x[0]):
    print(f"{float(t_start):6.1f}  {action}  (duration={duration})")

print()
print("Metrics:", res.metrics)

with PlanValidator(problem_kind=temporal_flight.kind) as v:
    print("Valid   :", bool(v.validate(temporal_flight, res.plan)))

Status   : SOLVED_SATISFICING
Plan kind: PlanKind.TIME_TRIGGERED_PLAN

Schedule (sorted by start time):
   0.0  fly(A, B)  (duration=10)
  10.0  fly(B, D)  (duration=10)
  20.0  fly(D, F)  (duration=10)
  30.0  fly(F, H)  (duration=10)

Metrics: {'goal_depth': '4', 'expanded_states': '5'}
Valid   : True


## 3 · Advanced Usage

To go beyond the defaults, configure the search via `params={"search": SearchParams(...)}`
(single queue) or `params={"search": MultiqueueParams(...)}` (multi queue).
This section first details single-queue tuning, then multi-queue.

### 3.1 · Single Queue

A `SearchParams` instance configures a single search queue: one search algorithm
running over one heuristic. Pass it to `OneshotPlanner` via
`params={"search": SearchParams(...)}`.

We tour the options in three groups:

- **§3.1.1 Heuristics** — `heuristic`, `inadmissible_numeric_heuristic_variant`, custom heuristics
- **§3.1.2 Search Algorithms** — `search`, `weight`
- **§3.1.3 Search-Control Flags** — the remaining boolean toggles

A full `SearchParams` field table is given at the end of §3.1 as a quick reference.

#### 3.1.1 · Heuristics

`SearchParams.heuristic` selects the heuristic function:

- `"hff"`
- `"hadd"`
- `"hmax"`
- `"blind"`
- `"custom"`

**Numeric (Warehouse)** — `wastar` + varying heuristic:

In [9]:
# A simple admissible custom heuristic: stock still needed (lower bound).
def my_heuristic(state):
    stock_val = state.get_value(stock()).constant_value()
    return float(max(0, 6 - stock_val))


results = []
for h in ["hff", "hadd", "hmax", "blind"]:
    r = solve_and_summarize(
        warehouse,
        SearchParams(search="wastar", heuristic=h, weight=0.8),
        label=h,
    )
    results.append(r)

# Custom heuristic — pass via heuristic_fn=, set params heuristic="custom"
results.append(
    solve_and_summarize(
        warehouse,
        SearchParams(search="wastar", heuristic="custom", weight=0.8),
        heuristic_fn=my_heuristic,
        label="custom (stock-gap)",
    )
)

show_table(results)

label              | status             | expanded | depth | plan_len | time_s
-------------------+--------------------+----------+-------+----------+-------
hff                | SOLVED_SATISFICING | 50       | 16    | 16       | 0.011 
hadd               | SOLVED_SATISFICING | 21       | 16    | 16       | 0.011 
hmax               | SOLVED_SATISFICING | 21       | 16    | 16       | 0.010 
blind              | SOLVED_SATISFICING | 55       | 16    | 16       | 0.010 
custom (stock-gap) | SOLVED_SATISFICING | 23       | 16    | 16       | 0.010 


**Takeaway:** `hadd` and `hmax` (~21 states) cut expansions in half versus `hff` (50) and `blind` (55) on this numeric warehouse — the additive/max relaxations exploit the numeric structure more sharply than `hff`'s relaxed plan, which barely beats no heuristic at all. The custom `stock-gap` heuristic lands right alongside `hadd`/`hmax` at 23.

##### `inadmissible_numeric_heuristic_variant`

The relaxed-plan heuristics (`hff`, `hadd`, `hmax`) take a shortcut on **constant-assignment** effects (`x := c`) and **complex numeric** effects: any operator with such an effect on a goal-relevant fluent is treated as achieving the condition in one application — an admissible lower bound.

Setting the flag to `True` disables that shortcut. Assignment / complex effects then contribute a fixed `1.0` net-effect, so the heuristic typically estimates more repetitions are needed. The resulting h-values may overestimate true cost (no longer a lower bound), but the higher estimates often guide weighted A* to expand fewer states on numeric domains. The cell below demonstrates this on **fo-counters** (IPC 2023 numeric), whose `increment(c)` action increases a counter by its `rate_value(c)` — a fluent-dependent (complex) increase.

In [10]:
# Reuses the `counters_problem` defined in §1.2.
r_adm = solve_and_summarize(
    counters_problem,
    SearchParams(heuristic="hadd", inadmissible_numeric_heuristic_variant=False),
    label="admissible (default)",
)
r_inad = solve_and_summarize(
    counters_problem,
    SearchParams(heuristic="hadd", inadmissible_numeric_heuristic_variant=True),
    label="inadmissible",
)
show_table([r_adm, r_inad])

label                | status             | expanded | depth | plan_len | time_s
---------------------+--------------------+----------+-------+----------+-------
admissible (default) | SOLVED_SATISFICING | 380      | 9     | 9        | 0.047 
inadmissible         | SOLVED_SATISFICING | 119      | 9     | 9        | 0.027 


**Takeaway:** the inadmissible variant cuts the expanded states by ~3× on this fo-counters instance (with `hadd`). The flag is most useful in numeric domains whose actions have complex (fluent-dependent) or assignment effects on goal-relevant fluents — exactly the case the admissible shortcut over-simplifies. `hff` is unaffected (it doesn't take the same shortcut); `hadd` and `hmax` are.

##### Custom heuristic

Pass any `Callable[[State], Optional[float]]` as `heuristic=` to `solve()` (or as `heuristic_fn=` to our `solve_and_summarize` helper). The state argument exposes `state.get_value(fluent())` — pass the fluent expression (with parentheses for a 0-arity fluent); call `.constant_value()` on the returned UP `FNode` to get the Python value. When `heuristic="custom"`, the default weight is `1.0`.

The `my_heuristic` defined just above (a stock-gap lower bound) is the last row in the heuristics table — it expands fewer states than the built-in `hff` on this Warehouse.

In [11]:
# Minimal API: pass the callable directly to planner.solve(heuristic=…)
params = SearchParams(search="wastar", heuristic="custom", weight=0.8)
with OneshotPlanner(name="tamerlite", params={"search": params}) as planner:
    res = planner.solve(warehouse, heuristic=my_heuristic)

print("Status :", res.status.name)
print("Metrics:", res.metrics)

Status : SOLVED_SATISFICING
Metrics: {'goal_depth': '16', 'expanded_states': '23'}


#### 3.1.2 · Search Algorithms

`SearchParams.search` selects the algorithm:

| Value | Description |
|-------|-------------|
| `"wastar"` | Weighted A\* *(default)*; `weight` ∈ [0, 1] |
| `"astar"` | A\* (optimal; `weight` forced to 0.5) |
| `"gbfs"` | Greedy best-first search |
| `"bfs"` | Breadth-first (uninformed) |
| `"ehc"` | Enforced hill-climbing; incomplete search |

**`weight`** (`float`, default `0.8`): `1.0` = pure greedy, `0.5` = A*.

**Warehouse** — `hff` + varying search:

In [12]:
results = []
for s in ["wastar", "astar", "gbfs", "bfs", "ehc"]:
    r = solve_and_summarize(
        warehouse,
        SearchParams(search=s, heuristic="hff"),
        label=s,
    )
    results.append(r)

show_table(results)

label  | status             | expanded | depth | plan_len | time_s
-------+--------------------+----------+-------+----------+-------
wastar | SOLVED_SATISFICING | 50       | 16    | 16       | 0.011 
astar  | SOLVED_SATISFICING | 54       | 16    | 16       | 0.011 
gbfs   | SOLVED_SATISFICING | 35       | 16    | 16       | 0.010 
bfs    | SOLVED_SATISFICING | 12215    | 16    | 16       | 0.051 
ehc    | SOLVED_SATISFICING | 6525     | 16    | 16       | 0.034 


**Takeaway:** `bfs` ignores `hff` and `ehc` uses it only locally — both explode in state count (12k+ and 6.5k). The heuristic-guided algorithms cluster tightly (35–54): `gbfs` is purely greedy and expands fewest; `astar` (`weight=0.5` under the hood) and `wastar` (default `weight=0.8`) trade a few more expansions for a bounded-suboptimal path.

#### 3.1.3 · Search-Control Flags

##### Compression-safe actions

An action is **compression-safe** when it has no intermediate conditions/effects, its end conditions are subsumed by its overall conditions, and its non-start effects don't conflict with any other action's conditions. When *all* actions are compression-safe, TamerLite compiles the temporal problem into an equivalent sequential one and reuses classical search — usually much faster.

Flag: `compression_safe_actions` (`bool`, default `True`).

In [13]:
r_safe = solve_and_summarize(
    temporal_flight,
    SearchParams(search="wastar", heuristic="hff", compression_safe_actions=True),
    label="compression_safe=True",
)
r_nosafe = solve_and_summarize(
    temporal_flight,
    SearchParams(search="wastar", heuristic="hff", compression_safe_actions=False),
    label="compression_safe=False",
)
show_table([r_safe, r_nosafe])

label                  | status             | expanded | depth | plan_len | time_s
-----------------------+--------------------+----------+-------+----------+-------
compression_safe=True  | SOLVED_SATISFICING | 5        | 4     | 4        | 0.050 
compression_safe=False | SOLVED_SATISFICING | 9        | 8     | 4        | 0.039 


**Takeaway:** with `compression_safe_actions=True`, the temporal flight is compiled to a sequential problem and reuses classical search; turning it off forces the temporal engine, which expands more states (more event interleavings to consider).

##### Weak equality

In temporal planning two different states are never equal. With `weak_equality=True`, states that share some similarities are considered equal — fewer re-expansions, but the search is **incomplete** (a plan can be missed). If no plan is found, TamerLite automatically retries with strong equality.

Flag: `weak_equality` (`bool`, default `False`).

In [14]:
r_weak = solve_and_summarize(
    temporal_flight,
    SearchParams(search="wastar", heuristic="hff", weak_equality=True),
    label="weak_equality=True",
)
r_strong = solve_and_summarize(
    temporal_flight,
    SearchParams(search="wastar", heuristic="hff", weak_equality=False),
    label="weak_equality=False",
)
show_table([r_weak, r_strong])

label               | status             | expanded | depth | plan_len | time_s
--------------------+--------------------+----------+-------+----------+-------
weak_equality=True  | SOLVED_SATISFICING | 5        | 4     | 4        | 0.050 
weak_equality=False | SOLVED_SATISFICING | 5        | 4     | 4        | 0.048 


**Takeaway:** weak equality changes how concurrent active events are compared. On this flight only one event is ever in flight at a time, so the two settings tie; the win comes in domains with overlapping durative actions.

##### Symmetry breaking

When a problem has truly **interchangeable objects** (same type, identical role in initial state and goals), TamerLite enforces a lexicographic ordering: actions on object `o_k` are only explored after the same action has been applied to its symmetric predecessor. This prunes whole symmetric subtrees. Neither of our example problems has equivalent objects, so we omit the A/B here.

Flag: `symmetry_breaking` (`bool`, default `True`).

##### `incomplete_memory_bounded_search`

Most relevant for **non-temporal** problems, where the relaxed state space can grow without bound. When set to `True`, `wastar`/`astar`/`gbfs` switch to memory-bounded variants that may drop states — making the search **incomplete** (a plan can be missed). Default: `False`.

In [15]:
# incomplete_memory_bounded_search — A/B (numeric problem, no temporal state needed)
r_full = solve_and_summarize(
    warehouse,
    SearchParams(
        search="wastar", heuristic="hff", incomplete_memory_bounded_search=False
    ),
    label="unbounded",
)
r_mem = solve_and_summarize(
    warehouse,
    SearchParams(
        search="wastar", heuristic="hff", incomplete_memory_bounded_search=True
    ),
    label="memory-bounded",
)
show_table([r_full, r_mem])

label          | status             | expanded | depth | plan_len | time_s
---------------+--------------------+----------+-------+----------+-------
unbounded      | SOLVED_SATISFICING | 50       | 16    | 16       | 0.013 
memory-bounded | SOLVED_SATISFICING | 50       | 16    | 16       | 0.026 


**Takeaway:** at this scale the memory bound never kicks in, so the two runs expand identical states; the bounded variant pays a small overhead for the bookkeeping. Use it only when un-bounded search would run out of memory.

##### Other flags

- **`internal_heuristic_cache`** (default `True`): cache the heuristic value per state. Disable only under tight memory pressure.
- **`early_termination`** (default `False`): stop on the first *generated* (not yet expanded) goal-satisfying successor. Faster, but plan quality may suffer.
- **`relevance_analysis`** (default `True`): backward reachability pass before search to drop actions that can't contribute to the goal. Disable only when its overhead exceeds the savings.

#### `SearchParams` — field summary

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `search` | `str \| None` | `"wastar"` | Search algorithm (§3.1.2) |
| `heuristic` | `str \| None` | `"hff"` | Heuristic function (§3.1.1) |
| `weight` | `float \| None` | `0.8` | `f = (1−w)·g + w·h`; `0.5`=A*, `1`=greedy, `0`=uniform-cost |
| `internal_heuristic_cache` | `bool` | `True` | Cache heuristic values per state |
| `inadmissible_numeric_heuristic_variant` | `bool` | `False` | Inadmissible numeric estimates |
| `early_termination` | `bool` | `False` | Stop on first *generated* (not yet expanded) goal state |
| `weak_equality` | `bool` | `False` | Weaker state equality; retries with strong equality on failure |
| `symmetry_breaking` | `bool` | `True` | Prune symmetric equivalent-object states |
| `compression_safe_actions` | `bool` | `True` | Contiguous expansion of compression-safe temporal actions |
| `relevance_analysis` | `bool` | `True` | Drop actions unreachable from the goal |
| `incomplete_memory_bounded_search` | `bool` | `False` | Memory-bounded variants of `wastar`/`astar`/`gbfs` |

### 3.2 · Multi Queue

`MultiqueueParams` runs several heuristic queues simultaneously (round-robin
node expansion). Useful when no single heuristic dominates.
Each entry in `queues` is a `HeuristicParams(heuristic=..., weight=...)`;
the global flags below apply to all queues uniformly.

A full `MultiqueueParams` field table is given at the end of §3.2 as a quick reference.

In [16]:
mq = MultiqueueParams(
    queues=[
        HeuristicParams(heuristic="hff", weight=0.8),
        HeuristicParams(heuristic="hadd", weight=0.8),
        HeuristicParams(heuristic="hmax", weight=0.5),
    ],
    compression_safe_actions=True,
)

r_single = solve_and_summarize(
    temporal_flight,
    SearchParams(search="wastar", heuristic="hff", weight=0.8),
    label="single  hff",
)
r_multi = solve_and_summarize(temporal_flight, mq, label="multiqueue hff+hadd+hmax")
show_table([r_single, r_multi])

label                    | status             | expanded | depth | plan_len | time_s
-------------------------+--------------------+----------+-------+----------+-------
single  hff              | SOLVED_SATISFICING | 5        | 4     | 4        | 0.060 
multiqueue hff+hadd+hmax | SOLVED_SATISFICING | 7        | 4     | 4        | 0.050 


#### `MultiqueueParams` — field summary

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `queues` | `List[HeuristicParams]` | — | One `HeuristicParams` per heuristic queue |
| `internal_heuristic_cache` | `bool` | `True` | Cache heuristic values per state |
| `inadmissible_numeric_heuristic_variant` | `bool` | `False` | Inadmissible numeric estimates |
| `early_termination` | `bool` | `False` | Stop on first generated goal state |
| `weak_equality` | `bool` | `False` | Weaker state equality (temporal) |
| `symmetry_breaking` | `bool` | `True` | Symmetric-state pruning |
| `compression_safe_actions` | `bool` | `True` | Temporal→sequential compilation when all actions are safe |
| `relevance_analysis` | `bool` | `True` | Drop goal-irrelevant actions |

`HeuristicParams(heuristic=..., weight=...)` accepts any heuristic string from §3.1.1.

## 4 · Anytime Planning

Attach a quality metric and use `AnytimePlanner` to receive a *stream* of progressively better plans. TamerLite finds an initial plan, then iteratively adds a tighter cost constraint and re-solves until the problem becomes unsolvable — at which point the last plan is certified optimal.

Supported metrics:

| Metric | Description |
|--------|-------------|
| `MinimizeSequentialPlanLength()` | Fewest actions |
| `MinimizeExpressionOnFinalState(expr)` | Minimize a numeric expression at the goal |
| `MaximizeExpressionOnFinalState(expr)` | Maximize a numeric expression at the goal |
| `MinimizeActionCosts(costs, default)` | Weighted sum of action costs |
| `MinimizeMakespan()` | *(temporal only)* Minimize overall completion time |

We demo `MinimizeMakespan` on the Temporal Flight. With both slow and express flights available, the satisficing solve finds a 40-makespan plan first; anytime then tightens the deadline and re-solves, eventually reaching the all-express 20-makespan optimum.

**Temporal Flight** — optimize `MinimizeMakespan()` (earliest completion time). Watch the makespan drop across `INTERMEDIATE` rows:

In [17]:
temporal_opt = temporal_flight.clone()
temporal_opt.name = "temporal_opt"
temporal_opt.add_quality_metric(MinimizeMakespan())

# Greedy satisficing (weight=1.0) typically picks a 40-makespan slow path first;
# anytime then tightens the deadline iteratively until the express-only 20 is reached.
params = SearchParams(
    search="wastar", heuristic="hff", weight=1.0, compression_safe_actions=False
)

with AnytimePlanner(name="tamerlite", params={"search": params}) as planner:
    print(f"{'#':>3}  {'Status':<32}  {'Makespan':>10}")
    print("-" * 50)
    for i, res in enumerate(planner.get_solutions(temporal_opt, timeout=60)):
        if res.plan and res.plan.timed_actions:
            makespan = max(
                float(t + (d if d is not None else 0))
                for t, _, d in res.plan.timed_actions
            )
        else:
            makespan = None
        print(f"{i + 1:>3}  {res.status.name:<32}  {str(makespan):>10}")
        if res.status != PlanGenerationResultStatus.INTERMEDIATE:
            break

  #  Status                              Makespan
--------------------------------------------------
  1  INTERMEDIATE                           40.03
  2  INTERMEDIATE                           35.03
  3  INTERMEDIATE                           30.03
  4  INTERMEDIATE                           25.03
  5  INTERMEDIATE                           20.03
  6  SOLVED_OPTIMALLY                       20.03
